# Smart Library Book Recommendation System

College ML notebook for the existing Smart Library MERN application.

## Part 1 — Project Overview

**Objective.** Train a simple book recommender on public Book-Crossing data and use the saved book-to-book relationships in the existing library app.

**Recommendation problem.** Suggest books a student has not already borrowed or rated, from titles the library actually stocks.

**Datasets.** Book-Crossing (`Books.csv`, `Users.csv`, `Ratings.csv`) and BookCrossing with Themes (category, description, Theme). Raw files are unchanged.

**Identity.** Kaggle `User-ID` values are not MongoDB students. Kaggle data learns book-to-book similarity. Application students use their own library history at recommendation time.

**Approach.** Clean and explore the data, filter to a 5/5 explicit-rating subset, compare popularity, item–item CF, content TF-IDF, and a hybrid experiment, then save the chosen model as artifacts.

**Final model: Item–Item Collaborative Filtering** (cosine similarity on user rating patterns).


## Part 2 — Data Loading, Cleaning & EDA

### Libraries

Inspection and modeling use pandas, NumPy, matplotlib/seaborn, and later scikit-learn.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
print("pandas", pd.__version__)


# 3. Load Datasets

Paths are relative to this notebook (`ml/notebooks/`), so raw files are at `../data/raw/`.

Book-Crossing CSVs are comma-separated and commonly need `latin-1`. The themes file is **semicolon-separated**.


In [ ]:
RAW_DIR = Path("..") / "data" / "raw"
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Raw data folder not found at {RAW_DIR.resolve()}. "
        "Open/run this notebook with the working directory set to ml/notebooks/"
    )

BX_DIR = RAW_DIR / "book_crossing"
THEMES_DIR = RAW_DIR / "themes"

print("Raw root:", RAW_DIR.resolve())
print("Files found:")
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  {path.relative_to(RAW_DIR)}  ({size_mb:.1f} MB)")


In [ ]:
books = pd.read_csv(BX_DIR / "Books.csv", encoding="latin-1", low_memory=False)
users = pd.read_csv(BX_DIR / "Users.csv", encoding="latin-1")
ratings = pd.read_csv(BX_DIR / "Ratings.csv", encoding="latin-1")
themes = pd.read_csv(THEMES_DIR / "BookCrossingThemes.csv", sep=";", encoding="utf-8", low_memory=False)

datasets = {
    "books": books,
    "users": users,
    "ratings": ratings,
    "themes": themes,
}
for name, df in datasets.items():
    print(f"{name:8s}  rows={df.shape[0]:,}  cols={df.shape[1]}")


# 4. Dataset Overview

Shape, columns, samples, dtypes, and descriptive stats for each table.


In [ ]:
def overview(name, df, sample_rows=3):
    print("=" * 80)
    print(name.upper(), "shape", df.shape)
    print("columns:", list(df.columns))
    print("\ndtypes:")
    print(df.dtypes)
    print("\nhead:")
    display(df.head(sample_rows))
    print("describe (numeric / include all where useful):")
    display(df.describe(include="all").T.head(20))

overview("Book-Crossing Books.csv", books)
overview("Book-Crossing Users.csv", users)
overview("Book-Crossing Ratings.csv", ratings)
overview("BookCrossingThemes.csv", themes)


# 5. Data Quality Analysis

Missing values, full-row duplicates, identifier uniqueness, rating scales, and whether expected academic fields (ISBN, title, author, genre, description) actually exist.


In [ ]:
def quality_report(name, df):
    print("=" * 80)
    print(name)
    missing = df.isna().sum()
    missing_pct = (100 * missing / len(df)).round(2)
    miss_table = pd.DataFrame({"missing": missing, "pct": missing_pct})
    miss_table = miss_table[miss_table["missing"] > 0]
    if miss_table.empty:
        print("No pandas-null missing values.")
    else:
        display(miss_table)

    print("full-row duplicates:", int(df.duplicated().sum()))
    print("unique counts:")
    for col in df.columns:
        print(f"  {col}: {df[col].nunique(dropna=True):,}")

for name, df in datasets.items():
    quality_report(name, df)


In [ ]:
print("BOOK IDENTIFIERS AND METADATA FIELDS")
print()
print("Book-Crossing Books.csv")
print("  ISBN column:        ", "ISBN" in books.columns)
print("  title:              ", "Book-Title" in books.columns)
print("  author:             ", "Book-Author" in books.columns)
print("  genre/category:     ", any(c.lower() in {"genre", "category", "theme"} for c in books.columns))
print("  description:        ", any("desc" in c.lower() for c in books.columns))
print("  other metadata:     ", [c for c in books.columns if c not in {"ISBN", "Book-Title", "Book-Author"}])
print()
print("Themes file")
print("  ISBN column:        ", "ISBN" in themes.columns)
print("  title:              ", "Book-Title" in themes.columns)
print("  author:             ", "Book-Author" in themes.columns)
print("  category:           ", "category" in themes.columns)
print("  Theme:              ", "Theme" in themes.columns)
print("  description:        ", "description" in themes.columns)
print("  cleaned_description:", "cleaned_description" in themes.columns)
print()
print("Users.csv has no book fields; columns:", list(users.columns))
print("Ratings.csv columns:", list(ratings.columns))


In [ ]:
print("ISBN FORMAT")

def isbn_profile(label, series):
    s = series.astype(str).str.strip()
    print(f"\n{label}")
    print("  sample:", s.head(6).tolist())
    print("  length value counts:", s.str.len().value_counts().sort_index().to_dict())
    print("  all digits share:", round(s.str.match(r"^\d+$").mean(), 4))
    print("  contains X share:", round(s.str.contains("X", case=False).mean(), 4))
    print("  contains hyphen share:", round(s.str.contains("-", regex=False).mean(), 6))
    print("  unique:", f"{s.nunique():,}", "rows:", f"{len(s):,}")

isbn_profile("books.ISBN", books["ISBN"])
isbn_profile("ratings.ISBN", ratings["ISBN"])
isbn_profile("themes.ISBN", themes["ISBN"])

stripped = books["ISBN"].astype(str).str.strip()
print("\nBooks ISBN strings that change after strip():", int((books["ISBN"].astype(str) != stripped).sum()))
print("Books ISBN nunique raw vs stripped:", books["ISBN"].nunique(), "vs", stripped.nunique())


In [ ]:
print("RATINGS — Book-Crossing Ratings.csv")
print(ratings["Book-Rating"].describe())
print("\nvalue counts:")
print(ratings["Book-Rating"].value_counts().sort_index())
n0 = int((ratings["Book-Rating"] == 0).sum())
n_exp = int((ratings["Book-Rating"].between(1, 10)).sum())
print(f"\n0-ratings (often treated as implicit / unrated in Book-Crossing): {n0:,} ({100*n0/len(ratings):.1f}%)")
print(f"explicit 1-10 ratings: {n_exp:,} ({100*n_exp/len(ratings):.1f}%)")
print("min/max:", int(ratings["Book-Rating"].min()), int(ratings["Book-Rating"].max()))

print("\nRATINGS — themes file")
print(themes["Book-Rating"].describe())
print(themes["Book-Rating"].value_counts().sort_index())
print("themes rating min/max:", int(themes["Book-Rating"].min()), int(themes["Book-Rating"].max()))
print("themes 0-ratings:", int((themes["Book-Rating"] == 0).sum()))


In [ ]:
print("UNIQUES AND KEY COMPLETENESS")
print("Users table unique User-ID:", f"{users['User-ID'].nunique():,}")
print("Ratings unique User-ID:    ", f"{ratings['User-ID'].nunique():,}")
print("Ratings unique ISBN:       ", f"{ratings['ISBN'].nunique():,}")
print("Books unique ISBN:         ", f"{books['ISBN'].nunique():,}")
print("Books unique titles:       ", f"{books['Book-Title'].nunique():,}")
print("Books unique authors:      ", f"{books['Book-Author'].nunique():,}")
print("Themes unique User-ID:     ", f"{themes['User-ID'].nunique():,}")
print("Themes unique ISBN:        ", f"{themes['ISBN'].nunique():,}")
print("Themes unique titles:      ", f"{themes['Book-Title'].nunique():,}")
print("Themes rows (user-book events, not unique books):", f"{len(themes):,}")
print("Themes duplicate ISBN rows (expected if file is rating-level):", int(themes["ISBN"].duplicated().sum()))
print()
print("Books missing author:", int(books["Book-Author"].isna().sum()))
print("Books missing publisher:", int(books["Publisher"].isna().sum()))
print("Users missing Age:", int(users["Age"].isna().sum()), f"({100*users['Age'].isna().mean():.1f}%)")
print("Themes missing Age:", int(themes["Age"].isna().sum()), f"({100*themes['Age'].isna().mean():.1f}%)")
print("Themes missing category:", int(themes["category"].isna().sum()), f"({100*themes['category'].isna().mean():.1f}%)")
print("Themes missing Theme:", int(themes["Theme"].isna().sum()))
print("Themes missing description:", int(themes["description"].isna().sum()))
print("Books duplicate titles:", int(books["Book-Title"].duplicated().sum()))
print("Books duplicate title+author:", int(books.duplicated(subset=["Book-Title", "Book-Author"]).sum()))

year = books["Year-Of-Publication"].astype(str)
non_numeric_year = ~year.str.fullmatch(r"\d+")
print("Books Year-Of-Publication non-numeric rows:", int(non_numeric_year.sum()))
if non_numeric_year.any():
    print(year[non_numeric_year].unique().tolist())


# 6. Dataset Relationship Analysis

Test whether the two sources can be connected. **Raw ISBN strings are compared first.** Then a simple normalization (uppercase, strip, `zfill(10)`) is applied because the themes file often stores ISBNs **without leading zeros**.

A join is only treated as usable if the overlap numbers support it.


In [ ]:
def canon_isbn(series):
    s = series.astype(str).str.strip().str.upper()
    s = s.str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
    s = s.str.replace(r"\.0$", "", regex=True)
    return s


def overlap(a, b, label_a, label_b):
    sa, sb = set(a), set(b)
    both = sa & sb
    print(f"{label_a} unique={len(sa):,}  {label_b} unique={len(sb):,}  intersection={len(both):,}")
    if sb:
        print(f"  share of {label_b} found in {label_a}: {100*len(both)/len(sb):.2f}%")
    if sa:
        print(f"  share of {label_a} found in {label_b}: {100*len(both)/len(sa):.2f}%")
    return both


print("RAW ISBN (no padding)")
overlap(books["ISBN"].astype(str), ratings["ISBN"].astype(str), "books ISBN", "ratings ISBN")
overlap(books["ISBN"].astype(str), themes["ISBN"].astype(str), "books ISBN", "themes ISBN")
overlap(ratings["ISBN"].astype(str), themes["ISBN"].astype(str), "ratings ISBN", "themes ISBN")

print("\nNORMALIZED ISBN (upper/strip) WITHOUT zfill")
b_n, r_n, t_n = canon_isbn(books["ISBN"]), canon_isbn(ratings["ISBN"]), canon_isbn(themes["ISBN"])
overlap(b_n, r_n, "books", "ratings")
overlap(b_n, t_n, "books", "themes")
overlap(r_n, t_n, "ratings", "themes")

print("\nNORMALIZED ISBN WITH zfill(10)")
b_z, r_z, t_z = b_n.str.zfill(10), r_n.str.zfill(10), t_n.str.zfill(10)
overlap(b_z, r_z, "books zfill10", "ratings zfill10")
books_themes = overlap(b_z, t_z, "books zfill10", "themes zfill10")
overlap(r_z, t_z, "ratings zfill10", "themes zfill10")

print("\nRow-level coverage after zfill(10)")
print("ratings rows whose ISBN is in books:", f"{r_z.isin(set(b_z)).mean()*100:.2f}%")
print("themes rows whose ISBN is in books:", f"{t_z.isin(set(b_z)).mean()*100:.2f}%")
print("themes unique ISBNs matched to books:", f"{len(books_themes):,} / {t_z.nunique():,}")


In [ ]:
print("TITLE / AUTHOR OVERLAP (themes vs Book-Crossing books)")

def norm_text(series):
    return series.fillna("").astype(str).str.strip().str.lower()

title_overlap = overlap(norm_text(books["Book-Title"]), norm_text(themes["Book-Title"]), "book titles", "theme titles")
overlap(norm_text(books["Book-Author"]), norm_text(themes["Book-Author"]), "book authors", "theme authors")

book_key = norm_text(books["Book-Title"]) + "||" + norm_text(books["Book-Author"])
theme_key = norm_text(themes["Book-Title"]) + "||" + norm_text(themes["Book-Author"])
overlap(book_key, theme_key, "books title+author", "themes title+author")

print("\nUSER overlap")
overlap(users["User-ID"], ratings["User-ID"], "users table", "ratings users")
overlap(users["User-ID"], themes["User-ID"], "users table", "themes users")
overlap(ratings["User-ID"], themes["User-ID"], "ratings users", "themes users")

print("\nGrain of the themes file (not a unique-book catalog):")
print("  rows:", f"{len(themes):,}")
print("  unique User-ID+ISBN:", f"{themes[['User-ID','ISBN']].drop_duplicates().shape[0]:,}")
print("  unique ISBN:", f"{themes['ISBN'].nunique():,}")


# 7. Initial EDA

A small chart set: rating scales, activity, and metadata coverage. These charts describe the files; they are not model results.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x="Book-Rating", data=ratings, ax=axes[0], color="#4C6A92")
axes[0].set_title("Book-Crossing ratings (includes 0)")
axes[0].set_xlabel("Book-Rating")
axes[0].set_ylabel("Count")

sns.countplot(x="Book-Rating", data=themes, ax=axes[1], color="#6B8F71")
axes[1].set_title("Themes file ratings (no zeros observed)")
axes[1].set_xlabel("Book-Rating")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
isbn_counts = ratings.groupby("ISBN").size().sort_values(ascending=False).head(15)
books_key = pd.DataFrame({"ISBN_z": b_z, "Book-Title": books["Book-Title"]}).drop_duplicates("ISBN_z")
top_books = isbn_counts.rename("n_ratings").reset_index()
top_books["ISBN_z"] = canon_isbn(top_books["ISBN"]).str.zfill(10)
top_books = top_books.merge(books_key, on="ISBN_z", how="left")
top_books["label"] = top_books["Book-Title"].fillna(top_books["ISBN"]).str.slice(0, 40)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=top_books, y="label", x="n_ratings", ax=ax, color="#4C6A92")
ax.set_title("Most-rated ISBNs in Book-Crossing Ratings.csv")
ax.set_xlabel("Number of rating rows")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
display(top_books[["ISBN", "Book-Title", "n_ratings"]])


In [ ]:
user_counts = ratings.groupby("User-ID").size().sort_values(ascending=False)
top_users = user_counts.head(15)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=top_users.values, y=top_users.index.astype(str), ax=axes[0], color="#C47B5A")
axes[0].set_title("Most active users (all rating rows)")
axes[0].set_xlabel("Rating rows")
axes[0].set_ylabel("User-ID")

explicit = ratings[ratings["Book-Rating"] > 0]
exp_per_user = explicit.groupby("User-ID").size()
axes[1].hist(np.clip(exp_per_user, 0, 50), bins=25, color="#C47B5A")
axes[1].set_title("Explicit ratings per user (clipped at 50)")
axes[1].set_xlabel("Ratings / user")
axes[1].set_ylabel("Users")
plt.tight_layout()
plt.show()

print("median rating-rows per user (all):", float(user_counts.median()))
print("median explicit ratings per user:", float(exp_per_user.median()))
print("users with at least one explicit rating:", f"{exp_per_user.shape[0]:,}")


In [ ]:
n_books = len(books)
n_theme_books = len(set(t_z) & set(b_z))
n_theme_unique = t_z.nunique()

coverage = pd.DataFrame(
    {
        "field": [
            "ISBN (Books.csv)",
            "Title (Books.csv)",
            "Author (Books.csv)",
            "Genre in Books.csv",
            "Description in Books.csv",
            "Theme/category/description (themes unique ISBN)",
            "Themes ISBN matched to Books after zfill(10)",
        ],
        "books_covered": [
            n_books,
            int(books["Book-Title"].notna().sum()),
            int(books["Book-Author"].notna().sum()),
            0,
            0,
            n_theme_unique,
            n_theme_books,
        ],
    }
)
coverage["pct_of_Books_csv"] = (100 * coverage["books_covered"] / n_books).round(3)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=coverage, y="field", x="pct_of_Books_csv", ax=ax, color="#4C6A92")
ax.set_title("Metadata coverage vs Books.csv catalog size")
ax.set_xlabel("% of Books.csv rows")
plt.tight_layout()
plt.show()
display(coverage)


In [ ]:
theme_books = themes.drop_duplicates(subset=["ISBN"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
theme_counts = theme_books["Theme"].value_counts().head(12)
sns.barplot(x=theme_counts.values, y=theme_counts.index, ax=axes[0], color="#6B8F71")
axes[0].set_title("Theme distribution (unique ISBN in themes file)")
axes[0].set_xlabel("Books")

cat_counts = theme_books["category"].value_counts().head(12)
sns.barplot(x=cat_counts.values, y=cat_counts.index.astype(str), ax=axes[1], color="#6B8F71")
axes[1].set_title("category distribution (unique ISBN in themes file)")
axes[1].set_xlabel("Books")
plt.tight_layout()
plt.show()

print("Theme labels:", theme_books["Theme"].nunique())
print("category labels (incl. messy values):", theme_books["category"].nunique(dropna=True))
print("example messy category values:")
print(theme_books["category"].dropna().astype(str).loc[lambda s: s.str.startswith("[")].head(8).tolist())


# 8. Initial Findings

Measured from the files on disk. These are **not** training results.

## What is in the files

| File | Rows | Role |
| --- | ---: | --- |
| `book_crossing/Books.csv` | 271,360 | Catalog: ISBN, title, author, year, publisher, cover URLs |
| `book_crossing/Users.csv` | 278,858 | Kaggle users: User-ID, Location, Age |
| `book_crossing/Ratings.csv` | 1,149,780 | User–ISBN ratings, scale **0–10** |
| `themes/BookCrossingThemes.csv` | 70,622 | **Rating-level** rows (not unique books) plus category, description, Theme |

## Fields the Book-Crossing catalog does **not** have

`Books.csv` has **no genre, no theme, and no description**. Those fields exist only in the themes file.

## Ratings

- Book-Crossing: **0 is the majority class** (~62%). In this dataset, 0 is usually an implicit interaction, not a one-star rating. Explicit ratings are **1–10** (~433,671 rows).
- Themes file: ratings **1–10 only** (no zeros).
- Activity is **sparse**: median rating count per user is **1**.

## Identifiers / ISBN

- Books and ratings mostly use **10-character ISBN** strings (some `X` check digits).
- Themes ISBNs are often **shorter** because **leading zeros were dropped**.
- **Raw ISBN intersection with themes is low (~12% of theme ISBNs).** That is an identifier problem, not proof that the books are unrelated.
- After `zfill(10)`, **all 5,229 unique theme ISBNs match Books.csv and Ratings.csv**. Title/author overlap is also ~100% of theme titles. So a merge is **valid only after ISBN normalization**, and it attaches rich text to a **small subset** of the catalog (~1.9% of Books.csv).

## Duplicate books

- No full duplicate rows in the four tables.
- Same work can appear under **multiple ISBNs** (editions): tens of thousands of repeated titles in Books.csv.
- The themes file repeats ISBN by design (one row per user–book rating).

## User IDs

Theme `User-ID` values sit inside the Book-Crossing user table. These IDs are **Kaggle users**, not Smart Library MongoDB students.

## What the data actually supports (no final algorithm lock-in)

Supported by the observed tables:

1. **Collaborative / rating-based methods** on Book-Crossing **explicit** ratings (1–10), with zeros handled as implicit or dropped — this is the large signal (users × books).
2. **Content features** (category, Theme, description) for **5,229 books only**, after ISBN padding.
3. A **hybrid** approach is possible on that **overlap subset**, not on the full 271k catalog, unless missing content is left empty.

Not supported without extra work:

- Using Kaggle `User-ID` as an application student id.
- Assuming every library book has a genre/description in these files.
- Treating raw theme ISBNs as join keys without normalization.
- Deep learning (not justified by this college-project data scale/sparsity, and out of scope).

**Next ML step (after this inspection):** clean ISBNs, decide how to treat 0-ratings, restrict or impute content features, then choose a simple explicit-rating recommender plus optional content similarity on the theme-enriched subset. Algorithm choice stays open until that cleaning notebook section is done.


---

# Part 2 — Data Cleaning (no model training)

Part 1 inspected the raw files. This part **cleans** them into modeling-ready tables.

Still **no training**, no `.pkl` models, and no connection to MongoDB students. Kaggle `User-ID` values stay Kaggle IDs.

Cover-image URL columns are not loaded. They are not needed for recommendation features.


## 1. Data Loading


Reload a lean copy from `../data/raw/`:

- Book-Crossing CSVs: comma-separated, `latin-1` (common for this dataset)
- Themes: **semicolon**-separated, UTF-8


In [ ]:
RAW_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BOOK_COLS = ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]

books_raw = pd.read_csv(
    RAW_DIR / "book_crossing" / "Books.csv",
    encoding="latin-1",
    usecols=BOOK_COLS,
    low_memory=False,
)
users_raw = pd.read_csv(RAW_DIR / "book_crossing" / "Users.csv", encoding="latin-1")
ratings_raw = pd.read_csv(RAW_DIR / "book_crossing" / "Ratings.csv", encoding="latin-1")
themes_raw = pd.read_csv(
    RAW_DIR / "themes" / "BookCrossingThemes.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False,
)

print("Loaded (cover-image columns omitted from Books.csv)")
for name, df in {
    "books_raw": books_raw,
    "users_raw": users_raw,
    "ratings_raw": ratings_raw,
    "themes_raw": themes_raw,
}.items():
    print(f"  {name:12s} {df.shape[0]:>10,} rows x {df.shape[1]} cols")


## 2. ISBN Normalization


Themes often store ISBNs **without leading zeros**. Padding every string with `zfill(10)` would also corrupt **ISBN-13** values (13 digits).

Reusable rules:

1. String, strip, uppercase, drop hyphens/spaces
2. Drop a trailing `.0` only when the rest is digits (CSV float artifact)
3. Keep 13-digit ISBN-13 as-is
4. Keep valid ISBN-10 (`##########` or `#########X`)
5. Zero-pad **shorter numeric** ISBNs (or digit+`X`) to 10 characters
6. Leave other junk unpadded (do not force it into a fake ISBN)


In [ ]:
import re


def normalize_isbn(value):
    """Normalize one ISBN. Does not rewrite the raw CSV files."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return pd.NA
    text = str(value).strip().upper().replace("-", "").replace(" ", "")
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    if not text or text in {"NAN", "NONE", "<NA>"}:
        return pd.NA
    if re.fullmatch(r"\d{13}", text):
        return text
    if re.fullmatch(r"\d{10}", text) or re.fullmatch(r"\d{9}X", text):
        return text
    if re.fullmatch(r"\d{1,9}", text) or re.fullmatch(r"\d{1,8}X", text):
        return text.zfill(10)
    return text


def is_valid_isbn(value):
    if pd.isna(value):
        return False
    return bool(re.fullmatch(r"\d{10}|\d{9}X|\d{13}", str(value)))


books_raw["isbn"] = books_raw["ISBN"].map(normalize_isbn)
users_raw = users_raw.rename(columns={"User-ID": "user_id"})
ratings_raw["isbn"] = ratings_raw["ISBN"].map(normalize_isbn)
ratings_raw = ratings_raw.rename(columns={"User-ID": "user_id", "Book-Rating": "rating"})
themes_raw["isbn"] = themes_raw["ISBN"].map(normalize_isbn)

print("Valid ISBN-10/13 share")
print("  books  ", f"{books_raw['isbn'].map(is_valid_isbn).mean():.4%}")
print("  ratings", f"{ratings_raw['isbn'].map(is_valid_isbn).mean():.4%}")
print("  themes ", f"{themes_raw['isbn'].map(is_valid_isbn).mean():.4%}")
print(
    "Theme rows whose normalized ISBN is in Books:",
    f"{themes_raw['isbn'].isin(set(books_raw['isbn'].dropna())).mean():.2%}",
)


## 3. Ratings Cleaning


Book-Crossing uses **0–10**. A **0** is an implicit mark (user interacted, no explicit star). It is **not** a 1-star review.

Keep both tables:

- `ratings_explicit`: 1–10 only (supervised preference)
- `ratings_implicit`: zeros only (kept; not deleted)


In [ ]:
ratings_explicit = ratings_raw.loc[ratings_raw["rating"].between(1, 10)].copy()
ratings_implicit = ratings_raw.loc[ratings_raw["rating"] == 0].copy()
n_other = int((~ratings_raw["rating"].between(0, 10)).sum())

print("Explicit ratings (1-10):", f"{len(ratings_explicit):,}")
print("Implicit / zero ratings:", f"{len(ratings_implicit):,}")
print("Ratings outside 0-10:   ", n_other)
print()
print("Explicit unique users:", f"{ratings_explicit['user_id'].nunique():,}")
print("Explicit unique ISBNs:", f"{ratings_explicit['isbn'].nunique():,}")
print("Implicit unique users:", f"{ratings_implicit['user_id'].nunique():,}")
print("Implicit unique ISBNs:", f"{ratings_implicit['isbn'].nunique():,}")
print()
print("Explicit rating distribution:")
print(ratings_explicit["rating"].value_counts().sort_index())


## 4. Books Cleaning


Decisions (rows are **not** dropped unless the ISBN is invalid):

| Issue | Action | Why |
| --- | --- | --- |
| Missing author (2 rows) | Fill `Unknown Author` | Keep the book; do not invent a real name |
| Missing publisher (2 rows) | Fill `Unknown Publisher` | Same |
| Non-numeric year / year 0 / year &gt; 2026 / year &lt; 1000 | Year set to missing | Invalid years stay in the catalog; year is not required |
| Duplicate normalized ISBN | Keep the more complete row | Normalization can collapse editions |
| Not a valid ISBN-10/13 after normalize | Drop | Cannot join reliably |


In [ ]:
books_work = books_raw.copy()
books_work["title"] = books_work["Book-Title"].fillna("").astype(str).str.strip()
books_work["author"] = books_work["Book-Author"].fillna("Unknown Author").astype(str).str.strip()
books_work["publisher"] = books_work["Publisher"].fillna("Unknown Publisher").astype(str).str.strip()

year_num = pd.to_numeric(books_work["Year-Of-Publication"], errors="coerce")
valid_year = year_num.between(1000, 2026)
print("Publication years set to missing:", int((~valid_year).sum()), "of", f"{len(books_work):,}")
books_work["year"] = year_num.where(valid_year)

print("Duplicate normalized ISBN rows:", int(books_work.duplicated("isbn").sum()))

books_work["completeness"] = (
    books_work["title"].ne("").astype(int)
    + books_work["author"].ne("Unknown Author").astype(int)
    + books_work["publisher"].ne("Unknown Publisher").astype(int)
    + books_work["year"].notna().astype(int)
)
books_work = books_work.sort_values(["isbn", "completeness"], ascending=[True, False])
books_clean = (
    books_work.drop_duplicates("isbn", keep="first")
    .loc[lambda df: df["isbn"].map(is_valid_isbn)]
    .drop(columns=["completeness", "ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"])
    .reset_index(drop=True)
)

print("books_clean rows (one per valid ISBN):", f"{len(books_clean):,}")
print(books_clean.head(3))


## 5. Users Cleaning


**Age is not a recommendation feature.** It is cleaned only so it is not misleading in EDA.

- Ages outside **5–100** (including 0 and 244) → missing
- Already-missing ages stay missing (no mean/median fill)
- These `user_id` values are **Kaggle IDs**, not Smart Library accounts


In [ ]:
users_clean = users_raw.copy()
age_num = pd.to_numeric(users_clean["Age"], errors="coerce")
users_clean["age"] = age_num.where(age_num.between(5, 100))
users_clean = users_clean.drop(columns=["Age"]).rename(columns={"Location": "location"})

n_invalid_age = int(age_num.notna().sum() - users_clean["age"].notna().sum())
print("Users:", f"{len(users_clean):,}")
print("Age values set to missing (invalid range):", n_invalid_age)
print("Age still present:", f"{users_clean['age'].notna().sum():,}")
print("Age still missing:", f"{users_clean['age'].isna().sum():,}")
print("Age will not be passed into the recommender.")


## 6. Themes Cleaning


Clean text **in memory**. Missing category stays missing (no invented genre).

`['Fiction']` style values are unquoted to `Fiction`. Descriptions and Theme labels are whitespace-normalized. ISBN uses the same `normalize_isbn` function.


In [ ]:
def clean_category(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text:
        return pd.NA
    match = re.fullmatch(r"\['(.+)'\]", text)
    if match:
        text = match.group(1).strip()
    text = re.sub(r"\s+", " ", text)
    return text if text else pd.NA


def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value).strip())
    return text if text else pd.NA


themes_work = themes_raw.copy()
themes_work["category_clean"] = themes_work["category"].map(clean_category)
themes_work["description_clean"] = themes_work["description"].map(clean_text)
themes_work["cleaned_description"] = themes_work["cleaned_description"].map(clean_text)
themes_work["Theme"] = themes_work["Theme"].map(clean_text)
themes_work = themes_work.loc[themes_work["isbn"].map(is_valid_isbn)].copy()

print("Theme rows after valid ISBN filter:", f"{len(themes_work):,}")
print("Unique ISBNs:", f"{themes_work['isbn'].nunique():,}")
print("Missing category (row-level):", int(themes_work["category_clean"].isna().sum()))
print("Missing Theme (row-level):", int(themes_work["Theme"].isna().sum()))
print("Missing description (row-level):", int(themes_work["description_clean"].isna().sum()))


## 7. Merge Analysis


Themes is **rating-level** (many rows per ISBN). Collapse to **one row per ISBN** before joining Books:

- themes → unique Theme labels, joined with `; `
- categories → unique cleaned categories, joined with `; `
- description / cleaned_description → first non-empty value

`enriched_books` = all cleaned books, with theme fields left missing when the ISBN was not in the themes subset. **No fabricated genres.**


In [ ]:
def join_unique(series):
    seen = []
    found = set()
    for value in series:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in found:
            found.add(text)
            seen.append(text)
    return "; ".join(seen) if seen else pd.NA


def first_non_empty(series):
    for value in series:
        if pd.notna(value) and str(value).strip():
            return value
    return pd.NA


themes_by_isbn = (
    themes_work.groupby("isbn", as_index=False)
    .agg(
        themes=("Theme", join_unique),
        categories=("category_clean", join_unique),
        description=("description_clean", first_non_empty),
        cleaned_description=("cleaned_description", first_non_empty),
        n_theme_ratings=("isbn", "size"),
    )
)

enriched_books = books_clean.merge(themes_by_isbn, on="isbn", how="left")

print("Unique books in enriched_books:", f"{len(enriched_books):,}")
print("With description:             ", f"{enriched_books['description'].notna().sum():,}")
print("With Theme:                   ", f"{enriched_books['themes'].notna().sum():,}")
print("With category:                ", f"{enriched_books['categories'].notna().sum():,}")
print("Without any theme metadata:   ", f"{enriched_books['themes'].isna().sum():,}")
print("Duplicate ISBNs in enriched:  ", int(enriched_books.duplicated("isbn").sum()))
display(enriched_books.loc[enriched_books["themes"].notna(), ["isbn", "title", "themes", "categories"]].head(5))


## 8. Interaction Dataset


Build `interactions` from **explicit** ratings only:

`user_id`, `isbn`, `rating`

Dropped if ISBN is invalid, the book is not in `books_clean`, the user is not in `users_clean`, or the pair is duplicated (keep last).

This table is **not** merged with MongoDB students.


In [ ]:
interactions = ratings_explicit.loc[ratings_explicit["isbn"].map(is_valid_isbn), ["user_id", "isbn", "rating"]].copy()
before = len(interactions)
interactions = interactions.drop_duplicates(["user_id", "isbn"], keep="last")
interactions = interactions.loc[interactions["isbn"].isin(books_clean["isbn"])]
interactions = interactions.loc[interactions["user_id"].isin(users_clean["user_id"])]
interactions["rating"] = interactions["rating"].astype(int)

print("Explicit rows before validity filters:", f"{before:,}")
print("Interaction rows after filters:       ", f"{len(interactions):,}")
print("Unique Kaggle users:                  ", f"{interactions['user_id'].nunique():,}")
print("Unique books:                         ", f"{interactions['isbn'].nunique():,}")
print("user_id is a Kaggle ID — not a MongoDB student id.")
display(interactions.head())


## 9. Basic EDA


Charts below use the **cleaned explicit interactions** and the **enriched book** table. They are for presentation, not model scores.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(x="rating", data=interactions, ax=ax, color="#4C6A92")
ax.set_title("Explicit rating distribution (1-10)")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of ratings")
plt.tight_layout()
plt.show()


In [ ]:
ratings_per_user = interactions.groupby("user_id").size()
ratings_per_book = interactions.groupby("isbn").size()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(np.clip(ratings_per_user, 0, 40), bins=20, color="#4C6A92")
axes[0].set_title("Ratings per user (clipped at 40)")
axes[0].set_xlabel("Ratings")
axes[0].set_ylabel("Users")
axes[1].hist(np.clip(ratings_per_book, 0, 40), bins=20, color="#6B8F71")
axes[1].set_title("Ratings per book (clipped at 40)")
axes[1].set_xlabel("Ratings")
axes[1].set_ylabel("Books")
plt.tight_layout()
plt.show()

print("Median ratings/user:", float(ratings_per_user.median()))
print("Median ratings/book:", float(ratings_per_book.median()))


In [ ]:
book_stats = (
    interactions.groupby("isbn")
    .agg(n_ratings=("rating", "size"), mean_rating=("rating", "mean"))
    .reset_index()
    .merge(books_clean[["isbn", "title", "author"]], on="isbn", how="left")
)

most_rated = book_stats.sort_values("n_ratings", ascending=False).head(12)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=most_rated, y="title", x="n_ratings", ax=ax, color="#4C6A92")
ax.set_title("Most-rated books (explicit ratings)")
ax.set_xlabel("Number of explicit ratings")
ax.set_ylabel("Title")
plt.tight_layout()
plt.show()

MIN_RATINGS_FOR_MEAN = 20
highest = (
    book_stats.loc[book_stats["n_ratings"] >= MIN_RATINGS_FOR_MEAN]
    .sort_values(["mean_rating", "n_ratings"], ascending=False)
    .head(12)
)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=highest, y="title", x="mean_rating", ax=ax, color="#C47B5A")
ax.set_title(f"Highest mean rating (at least {MIN_RATINGS_FOR_MEAN} explicit ratings)")
ax.set_xlabel("Mean rating")
ax.set_ylabel("Title")
ax.set_xlim(0, 10)
plt.tight_layout()
plt.show()


In [ ]:
meta_counts = pd.Series(
    {
        "All cleaned books": len(enriched_books),
        "With Theme": int(enriched_books["themes"].notna().sum()),
        "With category": int(enriched_books["categories"].notna().sum()),
        "With description": int(enriched_books["description"].notna().sum()),
    }
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=meta_counts.values, y=meta_counts.index, ax=ax, color="#4C6A92")
ax.set_title("Books with theme/category/description metadata")
ax.set_xlabel("Number of books")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

theme_books = enriched_books.loc[enriched_books["themes"].notna(), ["themes", "categories"]]
top_themes = theme_books["themes"].str.split("; ").explode().value_counts().head(12)
top_cats = theme_books["categories"].dropna().str.split("; ").explode().value_counts().head(12)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(x=top_themes.values, y=top_themes.index, ax=axes[0], color="#6B8F71")
axes[0].set_title("Top Theme labels (enriched books)")
axes[0].set_xlabel("Books")
axes[0].set_ylabel("Theme")
sns.barplot(x=top_cats.values, y=top_cats.index.astype(str), ax=axes[1], color="#6B8F71")
axes[1].set_title("Top categories (enriched books)")
axes[1].set_xlabel("Books")
axes[1].set_ylabel("Category")
plt.tight_layout()
plt.show()


## 10. Sparsity Analysis


Density = interactions / (users × books). Low density is why a naive full user–item matrix will not work well without filtering or a latent-factor method.


In [ ]:
def sparsity_report(df, label):
    n_users = df["user_id"].nunique()
    n_books = df["isbn"].nunique()
    n_int = len(df)
    density = n_int / (n_users * n_books) if n_users and n_books else 0.0
    per_user = df.groupby("user_id").size()
    per_book = df.groupby("isbn").size()
    out = {
        "subset": label,
        "users": n_users,
        "books": n_books,
        "interactions": n_int,
        "density": density,
        "sparsity": 1 - density,
        "avg_ratings_per_user": n_int / n_users if n_users else 0,
        "avg_ratings_per_book": n_int / n_books if n_books else 0,
        "median_ratings_per_user": float(per_user.median()) if len(per_user) else 0,
        "median_ratings_per_book": float(per_book.median()) if len(per_book) else 0,
    }
    return out


base_sparsity = sparsity_report(interactions, "explicit + valid ISBN in catalog")
display(pd.Series(base_sparsity))
print(
    f"Matrix density {base_sparsity['density']:.6%}  |  sparsity {base_sparsity['sparsity']:.4%}"
)


## 11. Filtering Strategy


The raw explicit matrix is large and sparse (median **1** rating per user and per book). Compare minimum-count thresholds **iteratively** (drop users, then books, repeat until stable). No threshold is locked in yet.

`(20, 20)` is included to show that an overly strict rule can collapse the set to empty.


In [ ]:
def filter_by_counts(df, min_user, min_book, max_rounds=15):
    out = df
    for _ in range(max_rounds):
        keep_users = out.groupby("user_id").size()
        keep_books = out.groupby("isbn").size()
        keep_users = keep_users[keep_users >= min_user].index
        keep_books = keep_books[keep_books >= min_book].index
        nxt = out.loc[out["user_id"].isin(keep_users) & out["isbn"].isin(keep_books)]
        if len(nxt) == len(out):
            break
        out = nxt
    return out


threshold_grid = [(1, 1), (5, 5), (10, 5), (10, 10), (20, 10), (20, 20)]
filter_rows = []
filtered_frames = {}
for min_user, min_book in threshold_grid:
    filtered = filter_by_counts(interactions, min_user, min_book)
    filtered_frames[(min_user, min_book)] = filtered
    rec = sparsity_report(filtered, f"min {min_user} ratings/user, {min_book}/book")
    rec["min_user"] = min_user
    rec["min_book"] = min_book
    filter_rows.append(rec)

filter_comparison = pd.DataFrame(filter_rows)[
    [
        "min_user",
        "min_book",
        "users",
        "books",
        "interactions",
        "density",
        "sparsity",
        "avg_ratings_per_user",
        "median_ratings_per_user",
        "avg_ratings_per_book",
        "median_ratings_per_book",
    ]
]
display(filter_comparison)

fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{r.min_user}/{r.min_book}" for r in filter_comparison.itertuples()]
ax.plot(labels, filter_comparison["interactions"], marker="o", color="#4C6A92")
ax.set_title("Interactions remaining vs min user/book thresholds")
ax.set_xlabel("min ratings per user / per book")
ax.set_ylabel("Interactions")
plt.tight_layout()
plt.show()

print("Candidate subsets (not trained): 5/5 (larger) and 10/10 (denser).")
print("20/10 is small; 20/20 is empty after iterative filtering.")


## 12. Save Processed Data


Write cleaned tables to `../data/processed/`. Raw CSVs are unchanged.

`ratings_explicit.csv` is the **filtered-valid interaction** table (`user_id`, `isbn`, `rating`), not the raw 0–10 file.

Implicit zeros stay in the notebook as `ratings_implicit` and are not written unless needed later.


In [ ]:
books_clean.to_csv(PROCESSED_DIR / "books_clean.csv", index=False)
users_clean.to_csv(PROCESSED_DIR / "users_clean.csv", index=False)
enriched_books.to_csv(PROCESSED_DIR / "enriched_books.csv", index=False)
interactions.to_csv(PROCESSED_DIR / "ratings_explicit.csv", index=False)
filter_comparison.to_csv(PROCESSED_DIR / "filter_threshold_comparison.csv", index=False)

print("Wrote:")
for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"  {path.name:40s} {path.stat().st_size / 1024:8.1f} KB")


## 13. Notebook Summary


Values below were **computed from the cleaned tables in this run**. They are not marketing numbers.

### Usable data (explicit interactions, valid ISBN in catalog)

| Item | Value |
| --- | ---: |
| Kaggle users with ≥1 explicit catalog rating | 68,116 |
| Books with ≥1 explicit rating | 149,706 |
| Explicit interaction rows | 384,029 |
| Cleaned catalog books (`books_clean`) | 270,928 |
| Enriched books (catalog + optional themes) | 270,928 |
| Books with description / Theme | 5,229 |
| Books with category | 4,984 |
| Implicit (zero) ratings kept in memory | 716,109 |

### Sparsity (unfiltered explicit interactions)

- Density ≈ **0.0038%** (sparsity ≈ **99.996%**)
- Average ratings/user ≈ **5.64**; **median 1**
- Average ratings/book ≈ **2.57**; **median 1**

### Filtering candidates (iterative min counts)

| Min user / book | Users | Books | Interactions | Density |
| --- | ---: | ---: | ---: | ---: |
| 1 / 1 | 68,116 | 149,706 | 384,029 | 0.0038% |
| **5 / 5** | 6,864 | 9,098 | 115,460 | 0.18% |
| 10 / 5 | 2,910 | 6,833 | 81,272 | 0.41% |
| **10 / 10** | 1,822 | 2,031 | 41,524 | 1.12% |
| 20 / 10 | 109 | 225 | 2,711 | 11.05% |
| 20 / 20 | 0 | 0 | 0 | — |

**5/5** keeps more data. **10/10** is denser and still large enough for a college CF experiment. **20/10** is too small; **20/20** is empty after iterative filtering.

### Limitations

- Theme/description/category exist for **~1.9%** of the catalog; content-based methods only apply there unless other books are left without text features.
- Rating **0** is implicit, not a star.
- Users are **Kaggle users**, not library students.
- Multiple editions (ISBNs) can be the same work.
- Age is noisy and **will not** drive recommendations.

**Not done in this part:** model training, evaluation, `.pkl` export, backend integration.


---

# Part 3 — Feature Engineering & Modeling

This part trains **simple, explainable** recommenders on the cleaned Kaggle data.

- Primary subset: users and books with **at least 5 explicit ratings** (iterative 5/5 filter from Part 2)
- Models: popularity baseline, item–item collaborative filtering, TF-IDF content-based (metadata books only)
- Hybrid only if enough 5/5 books actually have Theme/description/category
- Kaggle `user_id` values are **not** Smart Library students
- **No deep learning.** No saved `.pkl` yet (Part 4)

Evaluation language: **Hit Rate@10** is the share of users for whom the one held-out book appears in the top 10 suggestions. **Precision@10** is that hit counted over 10 slots (with one hidden book it is Hit Rate@10 / 10).


## Part 3 — Feature Engineering & Modeling

Primary subset: iterative **5/5** filter. Models: popularity baseline, **item–item collaborative filtering**, TF-IDF content-based, hybrid experiment.

**Hit@10** is the share of users whose held-out book appears in the top 10. It is not accuracy and is not the match percentage in the app.

### 5/5 filtered dataset


Load processed CSVs. Force `isbn` to string so leading zeros are not dropped. Rebuild the 5/5 subset with the same iterative rule as Part 2.


In [ ]:
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

PROCESSED_DIR = Path("..") / "data" / "processed"

books_clean = pd.read_csv(PROCESSED_DIR / "books_clean.csv", dtype={"isbn": str})
enriched_books = pd.read_csv(PROCESSED_DIR / "enriched_books.csv", dtype={"isbn": str})
ratings_explicit = pd.read_csv(
    PROCESSED_DIR / "ratings_explicit.csv",
    dtype={"user_id": int, "isbn": str, "rating": int},
)


def filter_by_counts(df, min_user, min_book, max_rounds=15):
    out = df
    for _ in range(max_rounds):
        n_user = out.groupby("user_id").size()
        n_book = out.groupby("isbn").size()
        nxt = out.loc[
            out["user_id"].isin(n_user[n_user >= min_user].index)
            & out["isbn"].isin(n_book[n_book >= min_book].index)
        ]
        if len(nxt) == len(out):
            break
        out = nxt
    return out.reset_index(drop=True)


modeling = filter_by_counts(ratings_explicit, 5, 5)
n_users = modeling["user_id"].nunique()
n_books = modeling["isbn"].nunique()
n_inter = len(modeling)
density = n_inter / (n_users * n_books)

print("Primary modeling set: iterative 5 ratings/user and 5 ratings/book")
print(f"  users         {n_users:,}")
print(f"  books         {n_books:,}")
print(f"  interactions  {n_inter:,}")
print(f"  matrix        {n_users:,} x {n_books:,}")
print(f"  density       {density:.6%}   sparsity {1-density:.4%}")

comp_1010 = filter_by_counts(ratings_explicit, 10, 10)
print(
    "10/10 comparison only (not used for training):",
    f"{comp_1010['user_id'].nunique():,} users,",
    f"{comp_1010['isbn'].nunique():,} books,",
    f"{len(comp_1010):,} interactions",
)

title_lookup = books_clean.drop_duplicates("isbn").set_index("isbn")["title"]


## 2. Train/Test Split


Leave-one-out: for every 5/5 user, hide **one** random rated book (`random_state=42`). All remaining ratings are training data. Similarity, popularity, and TF-IDF are fit on **training rows only**.


In [ ]:
rng = np.random.default_rng(42)
holdout_index = []
for _, group in modeling.groupby("user_id"):
    holdout_index.append(int(rng.choice(group.index.to_numpy())))

test_set = modeling.loc[holdout_index].copy()
train_set = modeling.drop(index=holdout_index).copy()

print("Train interactions:", f"{len(train_set):,}")
print("Test interactions: ", f"{len(test_set):,}", "(one per user)")
print("Train users/books:", train_set["user_id"].nunique(), "/", train_set["isbn"].nunique())
print("Overlap of test rows with train:", len(pd.merge(train_set, test_set, on=["user_id", "isbn"])))

rated_in_train = train_set.groupby("user_id")["isbn"].apply(set).to_dict()
test_by_user = test_set.set_index("user_id")["isbn"].to_dict()


## 3. Popularity Baseline


For each book in **training**, store mean rating and rating count. Rank by **count** (how often it was rated), then mean rating. That is a popularity baseline: recommend well-known books the user has not already rated in training.

The 5/5 filter already implies a minimum of 5 ratings in the modeling set; counts below are recomputed on train only.


In [ ]:
book_train_stats = (
    train_set.groupby("isbn")
    .agg(mean_rating=("rating", "mean"), n_ratings=("rating", "size"))
    .reset_index()
)
popularity_order = (
    book_train_stats.sort_values(["n_ratings", "mean_rating"], ascending=False)["isbn"]
    .tolist()
)


def recommend_popularity(user_id, n=10):
    seen = rated_in_train.get(user_id, set())
    recs = []
    for isbn in popularity_order:
        if isbn in seen:
            continue
        recs.append(
            {
                "isbn": isbn,
                "title": title_lookup.get(isbn, "(title missing)"),
                "score": float(book_train_stats.set_index("isbn").loc[isbn, "n_ratings"]),
            }
        )
        if len(recs) == n:
            break
    return pd.DataFrame(recs)


def hit_rate_at_k(recommend_isbns_fn, k=10):
    hits = 0
    n_eval = 0
    for user_id, held_isbn in test_by_user.items():
        recs = recommend_isbns_fn(user_id, k)
        n_eval += 1
        if held_isbn in recs:
            hits += 1
    hit_rate = hits / n_eval if n_eval else 0.0
    precision = hit_rate / k
    return {"hit_rate@10": hit_rate, "precision@10": precision, "hits": hits, "users": n_eval}


def popularity_isbns(user_id, k=10):
    seen = rated_in_train.get(user_id, set())
    out = []
    for isbn in popularity_order:
        if isbn not in seen:
            out.append(isbn)
        if len(out) == k:
            break
    return out


pop_metrics = hit_rate_at_k(popularity_isbns, k=10)
print("Popularity baseline (rank by training rating count)")
print(f"  Hit Rate@10   {pop_metrics['hit_rate@10']:.4%}")
print(f"  Precision@10  {pop_metrics['precision@10']:.4%}")
print(f"  hits / users  {pop_metrics['hits']} / {pop_metrics['users']}")
print("Hit Rate@10 = share of users whose hidden book appears in the 10 suggestions.")
display(recommend_popularity(next(iter(test_by_user)), n=5))


## 4. Item-Item Collaborative Filtering


Build a **user × book** matrix of training ratings. Books are similar if the same Kaggle users rated them similarly (**cosine** on book rating vectors).

To recommend for a user: take that user's training ratings, multiply by the book-similarity matrix, drop books already rated, keep the top 10.

`recommend_item_cf(user_id, n=10)` is the reusable function.


In [ ]:
train_users = train_set["user_id"].unique()
train_isbns = train_set["isbn"].unique()
user_index = {user_id: i for i, user_id in enumerate(train_users)}
isbn_index = {isbn: i for i, isbn in enumerate(train_isbns)}
index_to_isbn = {i: isbn for isbn, i in isbn_index.items()}

rating_matrix = csr_matrix(
    (
        train_set["rating"].astype(float),
        (
            train_set["user_id"].map(user_index),
            train_set["isbn"].map(isbn_index),
        ),
    ),
    shape=(len(user_index), len(isbn_index)),
)

item_similarity = cosine_similarity(rating_matrix.T, dense_output=True)
cf_scores = rating_matrix.dot(item_similarity)
train_nz = rating_matrix.tocoo()
cf_scores[train_nz.row, train_nz.col] = -np.inf

print("Rating matrix:", rating_matrix.shape)
print("Item similarity:", item_similarity.shape)


def recommend_item_cf(user_id, n=10):
    if user_id not in user_index:
        return pd.DataFrame(columns=["isbn", "title", "score"])
    row = cf_scores[user_index[user_id]]
    n_keep = min(n, np.isfinite(row).sum())
    top = np.argpartition(-row, kth=n_keep - 1)[:n_keep]
    top = top[np.argsort(-row[top])]
    rows = []
    for j in top:
        isbn = index_to_isbn[int(j)]
        rows.append(
            {
                "isbn": isbn,
                "title": title_lookup.get(isbn, "(title missing)"),
                "score": float(row[j]),
            }
        )
    return pd.DataFrame(rows)


def cf_isbns(user_id, k=10):
    rec = recommend_item_cf(user_id, n=k)
    return rec["isbn"].tolist()


cf_metrics = hit_rate_at_k(cf_isbns, k=10)
print("Item-item CF")
print(f"  Hit Rate@10   {cf_metrics['hit_rate@10']:.4%}")
print(f"  Precision@10  {cf_metrics['precision@10']:.4%}")
print(f"  hits / users  {cf_metrics['hits']} / {cf_metrics['users']}")
display(recommend_item_cf(next(iter(test_by_user)), n=5))


## 5. Content-Based Recommendation


Only books that actually have Theme, category, and/or `cleaned_description`. Combined text = those fields. **TF-IDF + cosine**. No embeddings.

`recommend_content(book_isbn, n=10)` returns similar books. For user-level evaluation, average similarity to the user's training books that have metadata.


In [ ]:
model_isbn_set = set(modeling["isbn"])
content_catalog = enriched_books.loc[enriched_books["isbn"].isin(model_isbn_set)].copy()


def nonempty(series):
    return series.notna() & series.astype(str).str.strip().ne("") & series.astype(str).str.strip().ne("nan")


has_theme = nonempty(content_catalog["themes"])
has_category = nonempty(content_catalog["categories"])
has_desc = nonempty(content_catalog["cleaned_description"])
content_catalog = content_catalog.loc[has_theme | has_category | has_desc].copy()
content_catalog["text"] = (
    content_catalog["themes"].fillna("").astype(str)
    + " "
    + content_catalog["categories"].fillna("").astype(str)
    + " "
    + content_catalog["cleaned_description"].fillna("").astype(str)
).str.strip()

tfidf = TfidfVectorizer(max_features=5000, min_df=2, stop_words="english")
content_tfidf = tfidf.fit_transform(content_catalog["text"])
content_similarity = cosine_similarity(content_tfidf)
content_isbns = content_catalog["isbn"].tolist()
content_index = {isbn: i for i, isbn in enumerate(content_isbns)}
content_isbn_set = set(content_isbns)

print("TF-IDF matrix:", content_tfidf.shape)
print("Vocabulary size:", len(tfidf.get_feature_names_out()))


def recommend_content(book_isbn, n=10):
    if book_isbn not in content_index:
        return pd.DataFrame(columns=["isbn", "title", "score"])
    i = content_index[book_isbn]
    sim = content_similarity[i].copy()
    sim[i] = -np.inf
    n_keep = min(n, np.isfinite(sim).sum())
    top = np.argpartition(-sim, kth=n_keep - 1)[:n_keep]
    top = top[np.argsort(-sim[top])]
    rows = []
    for j in top:
        isbn = content_isbns[int(j)]
        rows.append(
            {
                "isbn": isbn,
                "title": title_lookup.get(isbn, "(title missing)"),
                "score": float(sim[j]),
            }
        )
    return pd.DataFrame(rows)


def content_user_isbns(user_id, k=10):
    seen = rated_in_train.get(user_id, set())
    profile = [isbn for isbn in seen if isbn in content_isbn_set]
    if not profile:
        return []
    sim = np.mean(np.stack([content_similarity[content_index[isbn]] for isbn in profile]), axis=0)
    for isbn in seen:
        if isbn in content_index:
            sim[content_index[isbn]] = -np.inf
    n_keep = min(k, np.isfinite(sim).sum())
    top = np.argpartition(-sim, kth=n_keep - 1)[:n_keep]
    top = top[np.argsort(-sim[top])]
    return [content_isbns[int(j)] for j in top]


sample_content_isbn = content_isbns[0]
print("Example seed:", sample_content_isbn, title_lookup.get(sample_content_isbn, ""))
display(recommend_content(sample_content_isbn, n=5))


## 6. Content Coverage


Coverage is measured on **5/5 modeling books**, not the full 270k catalog. If too few modeling books have text, a hybrid would mostly copy CF and should be skipped.


In [ ]:
n_model_books = modeling["isbn"].nunique()
n_with_theme = int(has_theme.sum())
n_with_cat = int(has_category.sum())
n_with_desc = int(has_desc.sum())
n_with_any = len(content_catalog)
coverage = n_with_any / n_model_books

print("5/5 modeling books:", f"{n_model_books:,}")
print("  with Theme:              ", f"{n_with_theme:,}")
print("  with category:           ", f"{n_with_cat:,}")
print("  with cleaned_description:", f"{n_with_desc:,}")
print("  with any content field:  ", f"{n_with_any:,}", f"({coverage:.2%})")
print("CF books (in training matrix):", f"{len(isbn_index):,}")
print("Overlap CF train books ∩ content books:", f"{len(set(isbn_index) & content_isbn_set):,}")

HYBRID_MIN_COVERAGE = 0.20
hybrid_justified = coverage >= HYBRID_MIN_COVERAGE
print(
    "Hybrid justified:" if hybrid_justified else "Hybrid NOT justified:",
    f"content coverage {coverage:.2%} (threshold {HYBRID_MIN_COVERAGE:.0%})",
)


## 7. Hybrid Recommendation (only if justified)


If coverage is high enough: for each user, combine **normalized CF scores** with content similarity to that user's metadata books.

Default mix: **70% CF + 30% content**. One extra mix (80/20) is reported for comparison. These weights are not tuned as optimal.


In [ ]:
def hybrid_scores(cf_weight, content_weight):
    finite = np.isfinite(cf_scores)
    row_min = np.where(finite, cf_scores, np.inf).min(axis=1)
    row_max = np.where(finite, cf_scores, -np.inf).max(axis=1)
    span = np.maximum(row_max - row_min, 1e-9)
    cf_norm = (cf_scores - row_min[:, None]) / span[:, None]
    cf_norm[~finite] = 0.0

    train_c = train_set.loc[train_set["isbn"].isin(content_isbn_set)]
    content_term = np.zeros_like(cf_norm)
    if len(train_c):
        uc = csr_matrix(
            (
                np.ones(len(train_c)),
                (
                    train_c["user_id"].map(user_index),
                    train_c["isbn"].map(content_index),
                ),
            ),
            shape=(len(user_index), len(content_isbns)),
        )
        uc_norm = normalize(uc, norm="l1", axis=1)
        user_content_sim = uc_norm.dot(content_similarity)
        mapped_cols = []
        mapped_from = []
        for j, isbn in enumerate(content_isbns):
            if isbn in isbn_index:
                mapped_from.append(j)
                mapped_cols.append(isbn_index[isbn])
        content_term[:, np.array(mapped_cols)] = user_content_sim[:, np.array(mapped_from)]

    blended = cf_weight * cf_norm + content_weight * content_term
    blended[~finite] = -np.inf
    return blended


def isbns_from_score_matrix(score_matrix):
    def rec(user_id, k=10):
        if user_id not in user_index:
            return []
        row = score_matrix[user_index[user_id]]
        n_keep = min(k, np.isfinite(row).sum())
        top = np.argpartition(-row, kth=n_keep - 1)[:n_keep]
        top = top[np.argsort(-row[top])]
        return [index_to_isbn[int(j)] for j in top]

    return rec


hybrid_metrics = None
hybrid_metrics_80 = None
if hybrid_justified:
    hybrid_score_70 = hybrid_scores(0.7, 0.3)
    hybrid_metrics = hit_rate_at_k(isbns_from_score_matrix(hybrid_score_70), k=10)
    hybrid_score_80 = hybrid_scores(0.8, 0.2)
    hybrid_metrics_80 = hit_rate_at_k(isbns_from_score_matrix(hybrid_score_80), k=10)
    print("Hybrid 70% CF + 30% content")
    print(f"  Hit Rate@10   {hybrid_metrics['hit_rate@10']:.4%}")
    print(f"  Precision@10  {hybrid_metrics['precision@10']:.4%}")
    print(f"  hits / users  {hybrid_metrics['hits']} / {hybrid_metrics['users']}")
    print("Hybrid 80% CF + 20% content")
    print(f"  Hit Rate@10   {hybrid_metrics_80['hit_rate@10']:.4%}")
    print(f"  Precision@10  {hybrid_metrics_80['precision@10']:.4%}")
else:
    print("Skipping hybrid: content coverage on the 5/5 set is too low to mix in text similarity.")


## 8. Evaluation


Content-based Hit Rate@10 is computed only for users who have **at least one training book with metadata** (otherwise the content model has no profile).

All numbers come from the split above. A higher Hit Rate@10 means the hidden book landed in the top 10 more often.


In [ ]:
content_eval_users = [
    user_id
    for user_id in test_by_user
    if any(isbn in content_isbn_set for isbn in rated_in_train.get(user_id, set()))
]


def content_hit():
    hits = 0
    for user_id in content_eval_users:
        recs = content_user_isbns(user_id, k=10)
        if test_by_user[user_id] in recs:
            hits += 1
    n_eval = len(content_eval_users)
    hr = hits / n_eval if n_eval else 0.0
    return {"hit_rate@10": hr, "precision@10": hr / 10, "hits": hits, "users": n_eval}


content_metrics = content_hit()

eval_rows = [
    {
        "model": "Popularity (count, then mean)",
        "hit_rate@10": pop_metrics["hit_rate@10"],
        "precision@10": pop_metrics["precision@10"],
        "hits": pop_metrics["hits"],
        "users_evaluated": pop_metrics["users"],
    },
    {
        "model": "Item-item CF (cosine)",
        "hit_rate@10": cf_metrics["hit_rate@10"],
        "precision@10": cf_metrics["precision@10"],
        "hits": cf_metrics["hits"],
        "users_evaluated": cf_metrics["users"],
    },
    {
        "model": "Content TF-IDF (users with metadata profile)",
        "hit_rate@10": content_metrics["hit_rate@10"],
        "precision@10": content_metrics["precision@10"],
        "hits": content_metrics["hits"],
        "users_evaluated": content_metrics["users"],
    },
]
if hybrid_metrics is not None:
    eval_rows.append(
        {
            "model": "Hybrid 70% CF + 30% content",
            "hit_rate@10": hybrid_metrics["hit_rate@10"],
            "precision@10": hybrid_metrics["precision@10"],
            "hits": hybrid_metrics["hits"],
            "users_evaluated": hybrid_metrics["users"],
        }
    )
    eval_rows.append(
        {
            "model": "Hybrid 80% CF + 20% content",
            "hit_rate@10": hybrid_metrics_80["hit_rate@10"],
            "precision@10": hybrid_metrics_80["precision@10"],
            "hits": hybrid_metrics_80["hits"],
            "users_evaluated": hybrid_metrics_80["users"],
        }
    )

eval_table = pd.DataFrame(eval_rows)
eval_table["hit_rate@10"] = (eval_table["hit_rate@10"] * 100).round(3)
eval_table["precision@10"] = (eval_table["precision@10"] * 100).round(3)
eval_table = eval_table.rename(columns={"hit_rate@10": "Hit Rate@10 %", "precision@10": "Precision@10 %"})
display(eval_table)
print("Do not treat tiny gaps as a large real-world difference.")


## 9. Example Recommendations


Examples use **training** history only. Titles come from `books_clean.csv`.


In [ ]:
def show_history(user_id, n=5):
    hist = train_set.loc[train_set["user_id"] == user_id].sort_values("rating", ascending=False).head(n)
    hist = hist.merge(books_clean[["isbn", "title", "author"]], on="isbn", how="left")
    return hist[["isbn", "title", "author", "rating"]]


example_users = (
    train_set.groupby("user_id").size().sort_values(ascending=False).head(8).index.tolist()
)
picked = []
for user_id in example_users:
    if user_id in user_index:
        picked.append(int(user_id))
    if len(picked) == 3:
        break

for user_id in picked:
    print("=" * 80)
    print("Kaggle user_id", user_id, "(not a MongoDB student)")
    print("High training ratings:")
    display(show_history(user_id, n=5))
    print("Item-item CF top 10:")
    display(recommend_item_cf(user_id, n=10))
    if hybrid_justified:
        rec_fn = isbns_from_score_matrix(hybrid_score_70)
        hy = pd.DataFrame({"isbn": rec_fn(user_id, 10)})
        hy["title"] = hy["isbn"].map(title_lookup)
        print("Hybrid 70/30 top 10:")
        display(hy)

seed_from_user = show_history(picked[0], n=1)["isbn"].iloc[0]
print("=" * 80)
print("Content-based similar books for ISBN", seed_from_user)
print("Seed title:", title_lookup.get(seed_from_user, ""))
if seed_from_user in content_index:
    display(recommend_content(seed_from_user, n=10))
else:
    fallback = next(isbn for isbn in rated_in_train[picked[0]] if isbn in content_index)
    print("Seed has no metadata; using", fallback, title_lookup.get(fallback, ""))
    display(recommend_content(fallback, n=10))


### Part 3 result snapshot

Figures below were computed in this notebook.

| Item | Result |
| --- | --- |
| 5/5 modeling set | 6,864 users × 9,098 books, 115,460 ratings, density 0.185% |
| Train / test | 108,596 train ratings; 6,864 held-out ratings; overlap 0; seed 42 |
| Popularity | Hit@10 2.07%; Precision@10 0.21% |
| Item-item CF | Hit@10 **6.51%**; Precision@10 0.65% |
| Content TF-IDF | Hit@10 3.27%; Precision@10 0.33% |
| Hybrid 70/30 | Hit@10 6.67%; Precision@10 0.67% |
| Hybrid 80/20 | Hit@10 6.61%; Precision@10 0.66% |

Item–item CF improves over popularity. Hybrid gain is small, so **Item–Item CF** is the application model.


## Part 4 — Model Artifacts & Application Integration

The frozen model is 5/5 **item–item cosine CF**. Express loads `item_neighbors.json` at runtime (not the dense matrix). The UI match % is a relative score, not Hit@10.


### Saved artifacts and reload check

In [ ]:
from pathlib import Path
import json
import numpy as np

ARTIFACT_DIR = Path("..") / "artifacts"
sim_path = ARTIFACT_DIR / "item_similarity.npz"
index_path = ARTIFACT_DIR / "isbn_index.json"
neighbors_path = ARTIFACT_DIR / "item_neighbors.json"

print("item_similarity.npz exists:", sim_path.exists(), "bytes:", sim_path.stat().st_size if sim_path.exists() else 0)
print("isbn_index.json exists:", index_path.exists())
print("item_neighbors.json exists:", neighbors_path.exists())

isbn_index = json.loads(index_path.read_text(encoding="utf-8"))
neighbors = json.loads(neighbors_path.read_text(encoding="utf-8"))
loaded = np.load(sim_path)["item_similarity"]

print("isbn_index size:", len(isbn_index))
print("similarity shape:", loaded.shape)
print("runtime neighbor keys (library catalog):", len(neighbors))
print("sample neighbor list length:", len(next(iter(neighbors.values()))))


### Library catalog mapping

The original 10 seed books have no ISBN, so they cannot join the CF index. `library_catalog.csv` adds 50 ISBN titles that all sit in the frozen 5/5 index. The seed script imports that CSV in addition to the original 10 books.


In [ ]:
import pandas as pd

catalog_path = Path("..") / "data" / "processed" / "library_catalog.csv"
catalog = pd.read_csv(catalog_path, dtype={"isbn": str})
print("catalog rows:", len(catalog))
print("unique ISBNs:", catalog["isbn"].nunique())
print("ISBNs in frozen index:", catalog["isbn"].isin(isbn_index).sum())


### Application integration approach

1. Load the MongoDB student.
2. Collect borrowed and rated library books.
3. Map those books to ISBN.
4. Look up neighbors in `item_neighbors.json`.
5. Rank candidates and **filter to books in MongoDB**.
6. Return via existing `/api/recommendations` to Home and For You.

Kaggle User-ID is not mapped to MongoDB users. Students without usable ISBN history get a **popularity fallback**. Missing ISBNs are skipped (no crash).


## Part 5 — Final Results & Conclusion

### Final model

**Final Model: Item–Item Collaborative Filtering**

The model calculates cosine similarity between books from user rating patterns. Books similar to books a student has interacted with are ranked as recommendations.

The hybrid experiment is documented in Part 3. It is not the application model: the extra Hit@10 is small, and Item–Item CF is simpler to maintain and explain.

### Evaluation

| Model | Hit@10 | Precision@10 |
| --- | ---: | ---: |
| Popularity | 2.07% | 0.21% |
| Item-Item CF | **6.51%** | **0.65%** |
| Content TF-IDF | 3.27% | 0.33% |
| Hybrid 70/30 | 6.67% | 0.67% |
| Hybrid 80/20 | 6.61% | 0.66% |

Item–Item CF substantially improves over popularity (6.51% vs 2.07% Hit@10). Hybrid 70/30 is only slightly higher (6.67%).

### Limitations

- The application catalog is small.
- Books without ISBN cannot directly participate in Item-CF.
- Cold-start students use a popularity fallback.
- Book-Crossing data is external to the application’s real students.
- Recommendation quality depends on available interaction history.
- Content metadata covers only part of the modeling catalog.

### Conclusion

Real Book-Crossing data was cleaned and analyzed. Several simple recommenders were evaluated on a leave-one-out split. **Item–Item Collaborative Filtering** was selected, saved as runtime artifacts, and integrated into the existing MERN application. The app generates recommendations for students with usable ISBN history and uses a popularity fallback otherwise. This is a college project demonstration, not a production recommender.
